# 4 · Scenario simulator
Volume = (1+p)^e, revenue = (1+p)^(1+e), gross profit = (1+p)^e·(m+p)/m. Ranges come from the 95% interval of e. Only published elasticities, ±10% max (BR-10).

In [1]:
import os, sys
from pathlib import Path
# run from the repository root so relative paths work
if Path.cwd().name == "notebooks":
    os.chdir("..")
sys.path.insert(0, "src")
import pandas as pd, numpy as np
pd.set_option("display.width", 160, "display.max_columns", 20)

In [2]:
from simulator import simulate, classify
vol, rev, prof = simulate(-1.5, 0.05)
print(f'e=-1.5, +5%: volume {vol:+.2%}, revenue {rev:+.2%}, gross profit {prof:+.2%}')

e=-1.5, +5%: volume -7.06%, revenue -2.41%, gross profit +8.43%


## Why the threshold is 1/m
Gross profit rises with a small price increase while |e| < 1/m (3.3 at a 30% margin); revenue rises only while |e| < 1.

In [3]:
pd.DataFrame([(e, *simulate(e, 0.05)) for e in [-0.5, -1.0, -2.0, -3.0, -3.5, -4.0]],
             columns=['elasticity', 'volume', 'revenue', 'gross_profit']).round(3)

,elasticity,volume,revenue,gross_profit
0,-0.5,-0.024,0.025,0.139
1,-1.0,-0.048,0.000,0.111
2,-2.0,-0.093,-0.048,0.058
3,-3.0,-0.136,-0.093,0.008
4,-3.5,-0.157,-0.115,-0.016
5,-4.0,-0.177,-0.136,-0.040


## +5% on every published brand

In [4]:
g = pd.read_csv('data/output/scenario_grid.csv')
g[(g['price_change'].round(2) == 0.05) & (g['level'] == 'Brand')][['segment', 'volume_pct', 'volume_pct_low', 'volume_pct_high', 'revenue_pct', 'profit_pct', 'profit_delta']].round(3)

,segment,volume_pct,volume_pct_low,volume_pct_high,revenue_pct,profit_pct,profit_delta
36,Bag Snacks | Frito Lay,-0.071,-0.080,-0.063,-0.025,0.083,7591.986
57,Bag Snacks | Mksl,-0.070,-0.081,-0.059,-0.024,0.085,1907.794
78,Bag Snacks | Private Label,-0.029,-0.035,-0.023,0.019,0.133,24556.673
99,Bag Snacks | Shultz,-0.015,-0.020,-0.010,0.034,0.149,8205.475
120,Bag Snacks | Snyder's,-0.086,-0.092,-0.080,-0.040,0.067,6788.845
162,Cold Cereal | General Mills,-0.037,-0.039,-0.036,0.011,0.123,62962.267
183,Cold Cereal | Kellogg,-0.094,-0.098,-0.089,-0.049,0.057,23898.617
204,Cold Cereal | Post Foods,-0.088,-0.097,-0.080,-0.043,0.064,16583.255
225,Cold Cereal | Private Label,-0.068,-0.073,-0.064,-0.022,0.087,18791.690
267,Frozen Pizza | King,-0.137,-0.150,-0.124,-0.094,0.007,572.488


## Pricing stance (BR-11)

In [5]:
pd.read_csv('data/output/pricing_stance.csv')

,segment_id,level,segment,elasticity,ci_low,ci_high,stance
0,2,Category,Bag Snacks,-1.228,-1.308,-1.148,Moderate: price up grows profit but loses volu...
1,3,Brand,Bag Snacks | Frito Lay,-1.516,-1.707,-1.325,Moderate: price up grows profit but loses volu...
2,4,Brand,Bag Snacks | Mksl,-1.488,-1.734,-1.242,Moderate: price up grows profit but loses volu...
3,5,Brand,Bag Snacks | Private Label,-0.608,-0.737,-0.478,Low sensitivity: price increase candidate
4,6,Brand,Bag Snacks | Shultz,-0.310,-0.411,-0.209,Low sensitivity: price increase candidate
...,...,...,...,...,...,...,...
67,91,Product,Lstrne Cl Mint Antsptc Mw 1 lt,-1.892,-2.038,-1.746,Moderate: price up grows profit but loses volu...
68,92,Product,Lstrne Frs Brst Antspc Mw 1 lt,-1.992,-2.170,-1.815,Moderate: price up grows profit but loses volu...
69,93,Product,Post Hny Bn Ots Hny Rstd 18 oz,-1.703,-2.040,-1.366,Moderate: price up grows profit but loses volu...
70,94,Product,Post Fm Sz Hnybnch Ot Alm 18 oz,-1.722,-2.014,-1.430,Moderate: price up grows profit but loses volu...


## Price-pack opportunities

In [6]:
pd.read_csv('data/output/ppa.csv')[['category', 'manufacturer', 'description', 'size_oz', 'base_price_per_oz', 'price_index_per_oz', 'elasticity', 'opportunity']]

,category,manufacturer,description,size_oz,base_price_per_oz,price_index_per_oz,elasticity,opportunity
0,Bag Snacks,Frito Lay,Rldgld Braided Honey Wht 10 oz,10.000000,0.2929,220.7,-1.246,No action
1,Bag Snacks,Frito Lay,Rldgld Pretzel Sticks 16 oz,16.000000,0.1821,137.2,-1.742,No action
2,Bag Snacks,Frito Lay,Rldgld Tiny Twists Prtzl 16 oz,16.000000,0.1827,137.7,-1.632,No action
3,Bag Snacks,Mksl,Mksl Dutch Pretzels 16 oz,16.000000,0.1446,109.0,-1.150,No action
4,Bag Snacks,Mksl,Mksl Pretzel Sticks 16 oz,16.000000,0.1425,107.4,-1.561,No action
5,Bag Snacks,Mksl,Mksl Mini Twist Pretzels 16 oz,16.000000,0.1434,108.1,-1.753,No action
6,Bag Snacks,Private Label,Pl Pretzel Sticks 15 oz,15.000000,0.0922,69.5,-0.643,"Price up candidate: low sensitivity, below-ave..."
7,Bag Snacks,Private Label,Pl Twist Pretzels 15 oz,15.000000,0.0937,70.6,-0.509,"Price up candidate: low sensitivity, below-ave..."
8,Bag Snacks,Private Label,Pl Mini Twist Pretzels 15 oz,15.000000,0.0930,70.1,-0.678,"Price up candidate: low sensitivity, below-ave..."
9,Bag Snacks,Shultz,Shurgd Pretzel Rods 12 oz,12.000000,0.1706,128.6,-0.407,No action
